# Artificial Bee Colony for the TSP

Solves the berlin52 and eil101 TSPLIB instances with the Artificial Bee Colony algorithm (employed, onlooker and scout bee phases). Each instance runs 20 independent times with a stopping rule of 500 iterations.


In [1]:
import math
import random
import time

# =========================================
# 1) Read TSP file and extract coordinates
# =========================================
def read_tsp(filename):
    coords = []
    with open(filename, 'r') as f:
        start = False
        for line in f:
            if "NODE_COORD_SECTION" in line:
                start = True
                continue

            if "EOF" in line:
                break

            if start:
                parts = line.strip().split()
                if len(parts) == 3:
                    x = float(parts[1])
                    y = float(parts[2])
                    coords.append((x, y))

    return coords


# =========================================
# 2) Distance calculation between two cities
# =========================================
def distance(city1, city2):
    return math.sqrt((city1[0] - city2[0])**2 + (city1[1] - city2[1])**2)


# =========================================
# 3) Total distance of the route
# =========================================
def total_distance(route, coords):
    dist = 0
    for i in range(len(route)):
        current_city = coords[route[i]]
        next_city = coords[route[(i + 1) % len(route)]]
        dist += distance(current_city, next_city)
    return dist


# =========================================
# 4) Generate random route
# =========================================
def random_solution(n):
    route = list(range(n))
    random.shuffle(route)
    return route


# =========================================
# 5) Generate neighbor by swapping two cities
# =========================================
def neighbor(route):
    new_route = route[:]
    i, j = random.sample(range(len(route)), 2)
    new_route[i], new_route[j] = new_route[j], new_route[i]
    return new_route


# =========================================
# 6) Artificial Bee Colony Algorithm for TSP
# =========================================
def artificial_bee_colony(coords, iterations=500, food_sources=20, limit=50):
    n = len(coords)

    # Initialize food sources (solutions)
    foods = [random_solution(n) for _ in range(food_sources)]
    costs = [total_distance(route, coords) for route in foods]

    # Trial counter for scout bee phase
    trials = [0] * food_sources

    best_cost = min(costs)
    best_route = foods[costs.index(best_cost)][:]

    for iteration in range(iterations):

        # =========================================
        # Employed Bees Phase
        # Each bee tries to improve its current solution
        # =========================================
        for i in range(food_sources):
            new_route = neighbor(foods[i])
            new_cost = total_distance(new_route, coords)

            if new_cost < costs[i]:
                foods[i] = new_route
                costs[i] = new_cost
                trials[i] = 0
            else:
                trials[i] += 1

        # =========================================
        # Onlooker Bees Phase
        # Better solutions have higher chance to be selected
        # =========================================
        fitness = [1 / cost for cost in costs]
        total_fitness = sum(fitness)
        probabilities = [fit / total_fitness for fit in fitness]

        for _ in range(food_sources):
            selected = random.choices(range(food_sources), weights=probabilities, k=1)[0]

            new_route = neighbor(foods[selected])
            new_cost = total_distance(new_route, coords)

            if new_cost < costs[selected]:
                foods[selected] = new_route
                costs[selected] = new_cost
                trials[selected] = 0
            else:
                trials[selected] += 1

        # =========================================
        # Scout Bees Phase
        # Replace poor solutions that did not improve
        # =========================================
        for i in range(food_sources):
            if trials[i] >= limit:
                foods[i] = random_solution(n)
                costs[i] = total_distance(foods[i], coords)
                trials[i] = 0

        # Update best solution
        current_best = min(costs)
        if current_best < best_cost:
            best_cost = current_best
            best_route = foods[costs.index(current_best)][:]

    return best_cost


# =========================================
# 7) Run Algorithm B on small and large datasets
# =========================================

import os

# Look for the datasets in ../data (repository layout) or the current folder (Colab upload)
DATA_DIR = "../data" if os.path.isdir("../data") else "."
datasets = [os.path.join(DATA_DIR, name) for name in ["berlin52.tsp", "eil101.tsp"]]

for dataset in datasets:
    print("\n=========================================")
    print("Running Algorithm B (ABC) on:", dataset)
    print("=========================================")

    coords = read_tsp(dataset)

    results = []
    times = []

    # Run 20 independent runs
    for i in range(20):
        start_time = time.time()

        best_distance = artificial_bee_colony(coords, iterations=500)

        end_time = time.time()

        results.append(best_distance)
        times.append(end_time - start_time)

    print("All Results:")
    for i, r in enumerate(results):
        print(f"Run {i+1}: {r}")

    best = min(results)
    average = sum(results) / len(results)
    avg_time = sum(times) / len(times)

    print("Best Distance:", best)
    print("Average Distance:", average)
    print("Average Time:", avg_time)


Running Algorithm B (ABC) on: berlin52.tsp
All Results:
Run 1: 12776.772014389671
Run 2: 12859.625533826584
Run 3: 13382.145375154407
Run 4: 12876.97867919795
Run 5: 13383.011232242183
Run 6: 13214.015595315752
Run 7: 13383.289793698461
Run 8: 12822.492840081779
Run 9: 12590.251236944723
Run 10: 13491.441501022058
Run 11: 12843.556610984433
Run 12: 12899.731457002787
Run 13: 13584.688240142521
Run 14: 13518.77887874355
Run 15: 13841.986303113112
Run 16: 13091.596579804605
Run 17: 14062.049889271992
Run 18: 13251.165213464226
Run 19: 13478.473188840966
Run 20: 13040.31782357688
Best Distance: 12590.251236944723
Average Distance: 13219.618399340932
Average Time: 0.5842640399932861

Running Algorithm B (ABC) on: eil101.tsp
All Results:
Run 1: 1594.241817245013
Run 2: 1520.9679965470182
Run 3: 1519.1828757066405
Run 4: 1554.1462309781734
Run 5: 1609.6812289683544
Run 6: 1603.7311437692615
Run 7: 1415.8082823185414
Run 8: 1525.2284809601254
Run 9: 1505.1061612501599
Run 10: 1612.0242908656